# DiffDock blind docking — where does AMP want to bind OPLAH?

Autonomous binding-site discovery for the structure-to-screen pipeline (M4 prototype).
DiffDock places AMP on the **whole OPLAH surface with no predefined box** and ranks poses by
a learned confidence head. We then check locally whether it independently rediscovers the
adenine-F18 / monophosphate-P-loop footprint — no human telling it where to look.

**Runtime -> Change runtime type -> GPU (T4)**, then run top to bottom.
Upload `diffdock_inputs.zip` (contains `oplah.pdb`, `batch.csv`) when cell 4 asks.

> **Receptor note:** OPLAH is 1254 residues, but ESM-2 (DiffDock's receptor language model) caps at 1022. The `oplah.pdb` in `diffdock_inputs.zip` is therefore truncated to residues 6-1020 (1015 residues) — this **retains the entire AMP binding site** (all site residues are <=409) and only removes a C-terminal region >600 residues away from the pocket. The blind test is unaffected: the site is still a tiny patch on ~1000 residues of surface.


## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print("NO GPU - set Runtime -> GPU")

## 2. Clone DiffDock + install deps
MIT, github.com/gcorso/DiffDock. Installs take a few minutes.

In [ ]:
import os, subprocess, torch, importlib
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
if not os.path.isdir("/content/DiffDock"):
    subprocess.run("git clone https://github.com/gcorso/DiffDock.git /content/DiffDock", shell=True, check=True)
os.chdir("/content/DiffDock")

# 1) PyG compiled companions MUST match torch+CUDA exactly - use the official wheel index,
#    never a bare `pip install` (that source-compiles and ABI-mismatches).
tv = torch.__version__.split("+")[0]
cu = "cu" + (torch.version.cuda or "121").replace(".", "")
idx = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"
print("PyG wheel index:", idx)
subprocess.run(f"pip -q install torch_cluster torch_scatter torch_sparse torch_spline_conv -f {idx}", shell=True)

# 2) Everything else DiffDock imports, in ONE pass so we stop hitting missing modules one at a time.
#    Driven by the repo's own requirements.txt with torch* lines stripped (so Colab's GPU torch stays),
#    plus an explicit belt-and-suspenders list for anything the file omits.
subprocess.run(
    "grep -viE '^(torch|--|#|$)' requirements.txt 2>/dev/null "
    "| grep -viE 'torch[-_](geometric|scatter|sparse|cluster|spline)' > _req_notorch.txt; "
    "pip -q install -r _req_notorch.txt", shell=True)
subprocess.run("pip -q install torch-geometric rdkit e3nn fair-esm spyrmsd pandas biopython "
               "prody pyyaml scipy networkx posebusters", shell=True)

# 3) Import-verify the modules DiffDock actually loads at inference start.
need = ["torch_cluster","torch_scatter","torch_sparse","torch_geometric",
        "rdkit","e3nn","esm","spyrmsd","prody","yaml","scipy","networkx","Bio","pandas"]
missing=[m for m in need if importlib.util.find_spec(m) is None]
print("OK - all imports present" if not missing else f"STILL MISSING: {missing}")
print("DiffDock ready at", os.getcwd())

## 3. Apply the sandbox setrlimit fix
DiffDock raises the open-file limit past what a hosted runtime allows; clamp it to the hard limit.

In [ ]:
import re, pathlib
p = pathlib.Path("inference.py"); s = p.read_text()
s2 = re.sub(r"resource\.setrlimit\(resource\.RLIMIT_NOFILE, \([^)]*\)\)",
            "resource.setrlimit(resource.RLIMIT_NOFILE, (min(64000, resource.getrlimit(resource.RLIMIT_NOFILE)[1]), resource.getrlimit(resource.RLIMIT_NOFILE)[1]))", s)
p.write_text(s2)
print("patched" if s2!=s else "no match - check inference.py top if cell 6 errors on setrlimit")

## 4. Upload inputs

In [ ]:
from google.colab import files
import zipfile, glob, shutil, csv
up = files.upload()                    # pick diffdock_inputs.zip
z = next(iter(up))
with zipfile.ZipFile(z) as zf: zf.extractall("/content/dd_in")
pdb = glob.glob("/content/dd_in/**/oplah.pdb", recursive=True)[0]
csvp = glob.glob("/content/dd_in/**/batch.csv", recursive=True)[0]
shutil.copy(pdb, "/content/DiffDock/oplah.pdb")
rows=list(csv.DictReader(open(csvp)))
for r in rows: r["protein_path"]="oplah.pdb"
with open("/content/DiffDock/batch.csv","w",newline="") as fh:
    w=csv.DictWriter(fh,fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print("inputs staged:", rows)

## 5. Raise sample count (edit a COPIED YAML - CLI flags are ignored)
The config YAML overwrites argparse, so `--samples_per_complex` on the CLI does nothing.
Copy the YAML, edit the copy - 40 samples so AMP's preferred site is clear.

In [ ]:
import shutil, re, pathlib
shutil.copy("default_inference_args.yaml","blind.yaml")
y=pathlib.Path("blind.yaml"); t=y.read_text()
t=re.sub(r"samples_per_complex:\s*\d+","samples_per_complex: 40",t)
y.write_text(t)
print([l for l in t.splitlines() if "samples_per_complex" in l])

## 6. Run blind docking
First run is **silent for ~11 min** while DiffDock builds SO(3)/torus lookup tables - normal, not a hang.
Total ~15-20 min on a T4.

In [ ]:
import subprocess
r = subprocess.run("python3 -m inference --config blind.yaml --protein_ligand_csv batch.csv --out_dir out/",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("=== RETURN CODE", r.returncode, "- STDERR tail ===")
    print(r.stderr[-4000:])
    print("\nIf you see ModuleNotFoundError for torch_cluster/scatter/sparse, re-run cell 2 - "
          "the PyG wheel index install fixes it; a bare pip install does not.")

## 7. Package the ranked poses for download
All 40 ranked SDF poses (filenames carry the confidence logit).

In [ ]:
import glob, shutil, os
outd = [d for d in glob.glob("out/*") if os.path.isdir(d)][0]
sdfs = sorted(glob.glob(f"{outd}/rank*.sdf"))
print(f"{len(sdfs)} poses in {outd}; top:", os.path.basename(sdfs[0]) if sdfs else "NONE")
shutil.make_archive("/content/diffdock_poses","zip",outd)
from google.colab import files
files.download("/content/diffdock_poses.zip")